In [0]:
SELECT
  h.individual_id,
  h.individual_entity_id,
  h.tennant_id,
  t.tennant_name,
  ihn.is_trusted AS is_trusted_name,
  ihn.given1 AS given_name_1,
  ihn.given2 AS given_name_2,
  ihn.given3 AS given_name_3,
  ihn.given4 AS given_name_4,
  ihn.given5 AS given_name_5,
  ihn.family AS family_name,
  ihn.initials,
  ihn.is_preferred,
  ihn.use_code AS name_use_code,
  ihn.period_start AS name_used_from,
  ihn.period_end AS name_used_until,
  h.national_id,
  h.period_start AS hub_period_start,
  h.period_end AS hub_period_end,
  s.sat_individual_id,
  s.birth_date,
  s.birth_place_city,
  s.birth_place_country,
  s.is_deceased,
  s.deceased_datetime,
  s.gender_code,
  s.marital_status_code,
  s.ethnicity_code,
  s.religion_code,
  s.occupation_code,
  s.nationality_code,
  s.country_of_residence_code,
  s.is_employee,
  s.is_ex_employee,
  s.is_graduate,
  s.is_interpreter_required,
  s.is_employed,
  s.is_retired,
  s.is_student,
  s.is_individual_merged,
  s.period_start AS ind_sat_period_start,
  s.period_end AS ind_sat_period_end
FROM
  demo_banking_silver.qdp_individuals_all.hub_individual h
    JOIN demo_banking_silver.qdp_individuals_all.sat_individual s
      ON h.individual_id = s.individual_id
    JOIN demo_banking_silver.qdp_individuals_all.sat_human_name ihn
      ON h.individual_id = ihn.individual_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      on h.tennant_id = t.tennant_id
WHERE h.tennant_id = 100
--ORDER BY ihn.rank

In [0]:
SELECT * FROM demo_banking_silver.qdp_individuals_all.view_individual_addresses WHERE tennant_id = 100;

In [0]:
SELECT * FROM demo_banking_silver.qdp_individuals_all.view_individual_names WHERE tennant_id = 100;

In [0]:
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw ind;


In [0]:
%sql
-- 01. Create variables for use in this notebook


-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;



DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;




DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;


-- NEED TO CREATE DATA PRODUCT SCHEMAS

--02. Back out data from Tennant before populating with new data

--02-0. Remove all LINK records for this Tennant
DELETE FROM demo_banking_silver.qdp_links_all.link_individual_address_history WHERE tennant_id = 100;
DELETE FROM demo_banking_silver.qdp_links_all.link_organisation_address_history WHERE tennant_id = 100;

--02-1. Remove all Location Address records for this Tennant
DELETE FROM demo_banking_silver.qdp_locations_all.sat_address s
 WHERE s.address_id IN (
  SELECT h.address_id 
    FROM demo_banking_silver.qdp_locations_all.hub_address h 
    WHERE h.tennant_id = 100)
;


--02-2. Remove all Transaction records for this Tennant
DELETE FROM demo_banking_silver.qdp_transactions_all.sat_transaction s
 WHERE s.transaction_id IN (
  SELECT h.transaction_id 
    FROM demo_banking_silver.qdp_transactions_all.hub_transaction h 
    WHERE h.tennant_id = 100)
;




DELETE FROM demo_banking_silver.qdp_transactions_all.hub_transaction WHERE tennant_id = 100;


--02-3. Remove all Account records for this Tennant
DELETE FROM demo_banking_silver.qdp_accounts_all.sat_account s
  WHERE s.record_source_id = 100
-- WHERE s.account_id IN (
--  SELECT h.account_id 
--    FROM demo_banking_silver.qdp_accounts.hub_account h 
--    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = 100;


--02-4. Remove all Individual Customer records for this Tennant
DELETE FROM demo_banking_silver.qdp_individual_customers.sat_individual_customers s
 WHERE s.individual_customer_id IN (
  SELECT h.individual_customer_id
    FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers h 
    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_individual_customers.hub_individual_customers WHERE tennant_id = 100;



--02-5. Remove all Individual records for this Tennant
DELETE FROM demo_banking_silver.qdp_individuals_all.sat_human_name s
 WHERE s.record_source_id = 100
;

DELETE FROM demo_banking_silver.qdp_individuals_all.sat_individual s
 WHERE s.individual_id IN (
  SELECT h.individual_id 
    FROM demo_banking_silver.qdp_individuals_all.hub_individual h 
    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_individuals_all.hub_individual WHERE tennant_id = 100;



--02-6. Remove all Organisation Customer records for this Tennant
DELETE FROM demo_banking_silver.qdp_organisation_customers.sat_organisation_customers s
 WHERE s.organisation_customer_id IN (
  SELECT h.organisation_customer_id
    FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers h 
    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_organisation_customers.hub_organisation_customers WHERE tennant_id = 100;


--02-7. Remove all Organisation records for this Tennant
DELETE FROM demo_banking_silver.qdp_organisations_all.sat_organisation s
  WHERE s.record_source_id =  100
-- WHERE s.organisation_id IN (
--  SELECT h.organisation_id 
--    FROM demo_banking_silver.qdp_organisations_all.hub_organisation h 
--    WHERE h.tennant_id = 100)
;

DELETE FROM demo_banking_silver.qdp_organisations_all.hub_organisation WHERE tennant_id = 100;



-- Populate the location_all Data Product
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_location_raw;



-- 03-1. Insert into hub_address
INSERT INTO demo_banking_silver.qdp_locations_all.hub_address
(address_entity_id, tennant_id, period_start, period_end)
SELECT address_entity_id, 100, period_start, period_end
FROM demo_banking_bronze.risk_bcbs239_demo.risk_location_raw;

SELECT * FROM demo_banking_silver.qdp_locations_all.hub_address WHERE tennant_id = 100;


-- 03-2. Insert into sat_address
INSERT INTO demo_banking_silver.qdp_locations_all.sat_address
    (sat_address_id, 
     address_id, 
     load_datetime, 
     record_source_id, 
     postal_code,
     data_source_code,
--     data_source_concept_id,
--     data_source_raw_code,
--     data_source_raw_concept_id,
     type_code,
--     type_concept_id,
--     type_raw_code,
--     type_raw_concept_id,
     flatnumber_housenumber_housename,
     house_number,
     house_name,
     building_number,
     building_name,
     address_line1,
     address_line2,
     address_line3,
     address_line4,
     address_line5,
--     address_line6,
--     address_line7,
--     address_line8,
     street,
     district,
     city,
     county,
     state,
     country,
     country_code,
--     country_concept_id,
--     country_raw_code,
--     country_raw_concept_id,
     full_address,
--     uprn,
--     what3words,
     latitude,
     longitude,
--     directions,
     period_start,
     period_end
    )
SELECT
  monotonically_increasing_id() AS sat_address_id,
  h.address_id AS address_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
  q.postal_code AS postal_code,
  q.data_source_code AS data_source_code,
--  q.data_source_concept_id AS data_source_concept_id,
--  q.data_source_raw_code AS data_source_raw_code,
--  q.data_source_raw_concept_id AS data_source_raw_concept_id,
  q.type_code AS type_code,
--  q.type_concept_id AS type_concept_id,
--  q.type_raw_code AS type_raw_code,
--  q.type_raw_concept_id AS type_raw_concept_id,
  q.flatnumber_housenumber_housename AS flatnumber_housenumber_housename,
  q.house_number AS house_number,
  q.house_name AS house_name,
  q.building_number AS building_number,
  q.building_name AS building_name,
  q.address_line1 AS address_line1,
  q.address_line2 AS address_line2,
  q.address_line3 AS address_line3,
  q.address_line4 AS address_line4,
  q.address_line5 AS address_line5,
--  q.address_line6 AS address_line6,
--  q.address_line7 AS address_line7,
--  q.address_line8 AS address_line8,
  q.street AS street,
  q.district AS district,
  q.city AS city,
  q.county AS county,
  q.state AS state,
  q.country AS country,
  q.country_code AS country_code,
--  q.country_concept_id AS country_concept_id,
--  q.country_code_raw AS country_code_raw,
--  q.country_raw_concept_id AS country_raw_concept_id,
  q.full_address AS full_address,
--  q.uprn AS uprn,
--  q.what3words AS what3words,
  try_cast(q.latitude AS DOUBLE) AS latitude,
  try_cast(q.longitude AS DOUBLE) AS longitude,
--  q.directions AS directions,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_location_raw q
JOIN demo_banking_silver.qdp_locations_all.hub_address h
  ON h.address_entity_id = q.address_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_locations_all.sat_address WHERE record_source_id = 100;

--  04-0. Create individuals_all Data Product Entries
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw;


-- 04-1. Create the  hub_individual table records
INSERT INTO demo_banking_silver.qdp_individuals_all.hub_individual
(individual_entity_id, tennant_id, national_id, period_start, period_end)
SELECT individual_entity_id, 100, national_id, period_start, period_end
FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw  ind
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.hub_individual WHERE tennant_id = 100;

-- 04-2. Create the  sat_individual table records
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_individual
    ( 
     individual_id, 
     load_datetime, 
     record_source_id,
--     data_source_code,
--     data_source_concept_id,
--     data_source_raw_code,
--     data_source_raw_concept_id,
     birth_date,
--     birth_datetime,
--     birth_date_numeric,
--     estimated_birth_date,
--     estimated_birth_date_numeric,
--     is_multiple_birth,
--     multiple_birth_number,
     birth_place_city,
     birth_place_district,
     birth_place_country,
     birth_place_country_code,
--     photo,
--     is_deceased,
--     deceased_datetime,
--     deceased_date_numeric,
     gender_code,
--     gender_concept_id,
--     gender_raw_code,
--     gender_raw_concept_id,
     marital_status_code,
--     marital_status_concept_id,
--     marital_status_raw_code,
--     marital_status_raw_concept_id,
     ethnicity_code,
--     ethnicity_concept_id,
--     ethnicity_raw_code,
--     ethnicity_raw_concept_id,
--     religion_code,
--     religion_concept_id,
--     religion_raw_code,
--     religion_raw_concept_id,
     occupation_code,
--     occupation_concept_id,
--     occupation_raw_code,
--     occupation_raw_concept_id,
     nationality_code,
--     nationality_concept_id,
--     nationality_raw_code,
--     nationality_raw_concept_id,
     country_of_residence_code,
--     country_of_residence_concept_id,
--     country_of_residence_raw_code,
--     country_of_residence_raw_concept_id,
     primary_address_id,
     annual_income,
     number_of_dependants,
--     graduation_date,
     is_employee,
--     is_ex_employee,
     is_graduate,
--     is_interpreter_required,
--     is_employed,
--     is_retired,
--     is_student,
--     is_individual_merged,
     period_start,
     period_end 
    )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
--  q.data_source_code,
--  q.data_source_concept_id,
--  q.data_source_raw_code,
--  q.data_source_raw_concept_id,
  q.birth_date AS birth_date,
  --q.birth_datetime AS birth_datetime,
  --q.birth_date_numeric AS birth_date_numeric,
  --q.estimated_birth_date AS estimated_birth_date,
  --q.estimated_birth_date_numeric AS estimated_birth_date_numeric,
  --q.is_multiple_birth AS is_multiple_birth,
  --q.multiple_birth_number AS multiple_birth_number,
  q.birth_place_city AS birth_place_city,
  q.birth_place_district AS birth_place_district,
  q.birth_place_country AS birth_place_country,
  q.birth_place_country_code AS birth_place_country_code,
 -- q.photo AS photo,
 -- q.is_deceased AS is_deceased,
 -- q.deceased_datetime AS deceased_datetime,
 -- q.deceased_date_numeric AS deceased_date_numeric,
  q.gender_code AS gender_code,
--  q.gender_concept_id AS gender_concept_id,
--  q.gender_raw_code AS gender_raw_code,
--  q.gender_raw_concept_id AS gender_raw_concept_id,
  q.marital_status_code AS marital_status_code,
--  q.marital_status_concept_id AS marital_status_concept_id,
--  q.marital_status_raw_code AS marital_status_raw_code,
--  q.marital_status_raw_concept_id AS marital_status_raw_concept_id,
  q.ethnicity_code AS ethnicity_code,
--  q.ethnicity_concept_id AS ethnicity_concept_id,
--  q.ethnicity_raw_code AS ethnicity_raw_code,
--  q.ethnicity_raw_concept_id AS ethnicity_raw_concept_id,
--  q.religion_code AS religion_code,
--  q.religion_concept_id AS religion_concept_id,
--  q.religion_raw_code AS religion_raw_code,
--  q.religion_raw_concept_id AS religion_raw_concept_id,
  q.occupation_code AS occupation_code,
--  q.occupation_concept_id AS occupation_concept_id,
--  q.occupation_raw_code AS occupation_raw_code,
--  q.occupation_raw_concept_id AS occupation_raw_concept_id,
  q.nationality_code AS nationality_code,
--  q.nationality_concept_id AS nationality_concept_id,
--  q.nationality_raw_code AS nationality_raw_code,
--  q.nationality_raw_concept_id AS nationality_raw_concept_id,
  q.country_of_residence_code AS country_of_residence_code,
--  q.country_of_residence_concept_id AS country_of_residence_concept_id,
--  q.country_of_residence_raw_code AS country_of_residence_raw_code,
--  q.country_of_residence_raw_concept_id AS country_of_residence_raw_concept_id,
  q.primary_address_id AS primary_address_id,
  q.annual_income AS annual_income,
  q.number_of_dependants AS number_of_dependants,
--  q.graduation_date AS graduation_date,
  q.is_employee AS is_employee,
--  q.is_ex_employee AS is_ex_employee,
  q.is_graduate AS is_graduate,
--  q.is_interpreter_required AS is_interpreter_required,
--  q.is_employed AS is_employed,
--  q.is_retired AS is_retired,
--  q.is_student AS is_student,
--  q.is_individual_merged AS is_individual_merged,
  try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
  try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_individual WHERE record_source_id = 100;


-- 04-4-1. Create the table of Individual Human Names
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name (
    individual_id,
    load_datetime,
    record_source_id,
    rank,
    is_trusted,
    is_preferred,
    given1,
    given2,
    family,
    full_given,
    full_name,
    period_start,
    period_end)
  SELECT
    h.individual_id AS individual_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    try_cast(1 AS INT) AS rank,
    true AS is_trusted,
    true AS is_preferred,
    q.name_given1 AS given1,
    q.name_given2 AS given2,
    q.name_family AS family,
    '' AS full_given,
    q.name_full AS full_name,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
    ON h.individual_entity_id = q.individual_entity_id;
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name WHERE record_source_id = 100;

-- 04-4-1. Create the table of Individual Alias1 Human Names
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name (
    individual_id,
    load_datetime,
    record_source_id,
    rank,
    is_trusted,
    is_preferred,
    given1,
    given2,
    family,
    full_given,
    full_name,
    period_start,
    period_end)
  SELECT
    h.individual_id AS individual_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    try_cast(1 AS INT) AS rank,
    true AS is_trusted,
    true AS is_preferred,
    q.alias1_name_given1 AS given1,
    q.alias1_name_given2 AS given2,
    q.alias1_name_family AS family,
    '' AS full_given,
    q.alias1_name_full AS full_name,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
    ON h.individual_entity_id = q.individual_entity_id
  WHERE q.alias1_name_full IS NOT NULL;
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name WHERE record_source_id = 100;




-- 04-4-2. Create the table of Individual Alias2 Human Names
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name (
    individual_id,
    load_datetime,
    record_source_id,
    rank,
    is_trusted,
    is_preferred,
    given1,
    given2,
    family,
    full_given,
    full_name,
    period_start,
    period_end)
  SELECT
    h.individual_id AS individual_id,
    run_timestamp AS load_datetime,
    try_cast(100 AS BIGINT) AS record_source_id,
    try_cast(1 AS INT) AS rank,
    true AS is_trusted,
    true AS is_preferred,
    q.alias2_name_given1 AS given1,
    q.alias2_name_given2 AS given2,
    q.alias2_name_family AS family,
    '' AS full_given,
    q.alias2_name_full AS full_name,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end

  FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
    ON h.individual_entity_id = q.individual_entity_id
  WHERE q.alias2_name_full IS NOT NULL;
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name WHERE record_source_id = 100;



-- 04-4-3.  Create the table of associated Addresses for an Individual

INSERT INTO demo_banking_silver.qdp_links_all.link_individual_address_history (
    tennant_id,
    individual_id,
    address_id,
    is_primary_address,
    individual_entity_id,
    address_entity_id,
    use_code,
    period_start,
    period_end)
  SELECT
    try_cast(100 AS BIGINT) AS tennant_id,
     hi.individual_id AS individual_id,
     ha.address_id AS address_id,
    true AS is_primary_address,
     hi.individual_entity_id AS individual_entity_id,
     ha.address_entity_id AS address_entity_id,
    'Home Address' AS use_code,
    try_to_timestamp(default_start_date, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(default_end_date, 'dd-MM-yyyy') AS period_end
 FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
  JOIN demo_banking_silver.qdp_individuals_all.hub_individual hi
    ON hi.individual_entity_id = q.individual_entity_id
  JOIN demo_banking_silver.qdp_locations_all.hub_address ha
    ON ha.address_entity_id = q.primary_address_entity_id

--  FROM demo_banking_bronze.risk_bcbs239_demo.risk_individual_raw q
--  JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
--    ON h.individual_entity_id = q.individual_entity_id
--  WHERE q.alias2_name_full IS NOT NULL;
;

SELECT * FROM demo_banking_silver.qdp_links_all.link_individual_address_history WHERE tennant_id = 100;


-- 05-0. Create organisations_all Data Product Entries
SELECT * FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw;


-- 05-1. Create the  hub_organisation table records
INSERT INTO demo_banking_silver.qdp_organisations_all.hub_organisation
(organisation_entity_id, tennant_id, period_start, period_end)
SELECT organisation_entity_id, 100, period_start, period_end
FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
;

SELECT * FROM demo_banking_silver.qdp_organisations_all.hub_organisation WHERE tennant_id = 100;


-- 05-2. Create the  sat_organisation table records

INSERT INTO demo_banking_silver.qdp_organisations_all.sat_organisation
    ( 
     organisation_id, 
     load_datetime, 
     record_source_id,
     data_source_code,
     type_code,
     status_code,
     organisation_name,
--     trading_name,
--     duns_number,
--     organisation_description,
     organisation_level,
     parent_organisation_id,
     primary_adddress_id,
--     company_category_code,
--     company_resistration_name,
--     company_registration_number,
--     company_registration_date,
--     company_registration_country_code,
--     primary_operation_country_code,
--     vat_number,
--     is_customer,
--     risk_rating,
--     internal_credit_rating,
     period_start,
     period_end 
    )
SELECT
  h.organisation_id AS organisation_id,
  run_timestamp AS load_datetime,
  try_cast(100 AS BIGINT) AS record_source_id,
  q.data_source_code AS data_source_code,
  q.type_code AS type_code,
  q.status_code AS status_code,
  q.organisation_name AS organisation_name,
--  q.trading_name AS trading_name,
--  q.duns_number AS duns_number,
--  q.organisation_description AS organisation_description,
  try_cast(q.organisation_level AS INT) AS organisation_level,
  try_cast(0 AS BIGINT) AS parent_organisation_id,
  try_cast(q.primary_adddress_id as bigint) AS primary_adddress_id,
--  q.company_category_code AS company_category_code,
--  q.company_resistration_name AS company_registration_name,
--  q.company_registration_number AS company_registration_number,
--  q.company_registration_date AS company_registration_date,
--  q.company_registration_country_code AS compnay_registration_country_code,
--  q.primary_operation_country_code AS primary_oprtation_country_code,
--  q.vat_number AS vat_number,
--  q.is_customer AS is_customer,
--  q.risk_rating AS risk_rating,
--  q.internal_credit_rating AS internal_credit_rating,
  try_to_timestamp(q.period_start, 'dd-MM-yyyy') AS period_start,
  try_to_timestamp(q.period_end, 'dd-MM-yyyy') AS period_end

FROM demo_banking_bronze.risk_bcbs239_demo.risk_organisation_raw q
JOIN demo_banking_silver.qdp_organisations_all.hub_organisation h
  ON h.organisation_entity_id = q.organisation_entity_id;

SELECT * FROM demo_banking_silver.qdp_organisations_all.sat_organisation WHERE record_source_id = 100;


/**

-- 3. Create the table of individual_entity_id and associated Human Names
CREATE OR REPLACE TEMP VIEW individual_human_names AS
SELECT 
    individual_entity_id, 
    human_name.human_name.rank,
    human_name.human_name.is_trusted,
    human_name.human_name.is_preferred,
    human_name.human_name.use_code,
    try_cast(human_name.human_name.use_concept_id AS BIGINT) AS use_concept_id,
    human_name.human_name.use_raw_code,
    try_cast(human_name.human_name.use_raw_concept_id AS BIGINT) AS use_raw_concept_id,
    human_name.human_name.initials,
    human_name.human_name.given1,
    human_name.human_name.given1_soundex,
    human_name.human_name.given1_raw,
    human_name.human_name.given1_raw_soundex,
    human_name.human_name.given2,
    human_name.human_name.given2_soundex,
    human_name.human_name.given2_raw,
    human_name.human_name.given2_raw_soundex,
    human_name.human_name.given3,
    human_name.human_name.given3_soundex,
    human_name.human_name.given3_raw,
    human_name.human_name.given3_raw_soundex,
    human_name.human_name.given4,
    human_name.human_name.given4_soundex,
    human_name.human_name.given4_raw,
    human_name.human_name.given4_raw_soundex,
    human_name.human_name.given5,
    human_name.human_name.given5_soundex,
    human_name.human_name.given5_raw,
    human_name.human_name.given5_raw_soundex,
    human_name.human_name.family,
    human_name.human_name.family_soundex,
    human_name.human_name.family_raw,
    human_name.human_name.family_raw_soundex,
    human_name.human_name.full_given,
    human_name.human_name.full_given_soundex,
    human_name.human_name.full_name,
    human_name.human_name.full_name_soundex,
    human_name.human_name.full_name_raw,
    human_name.human_name.full_name_raw_soundex,
    human_name.human_name.full_name_parsed,
    try_to_timestamp(human_name.human_name.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(human_name.human_name.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT ind.individual_entity_id, explode(human_names) AS human_name 
  FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all ind
) AS exploded_names
LIMIT 1000;

SELECT * FROM individual_human_names;


-- 4. Create the table of individual_entity_id and associated Addresses
CREATE OR REPLACE TEMP VIEW individual_addresses AS
SELECT 
    individual_entity_id, 
    address.address.address_entity_id,
    address.address.rank,
    address.address.is_primary_address,
    address.address.use_code,
    try_cast(address.address.use_concept_id AS BIGINT) AS use_concept_id,
    address.address.use_raw_code,
    try_cast(address.address.use_raw_concept_id AS BIGINT) AS use_raw_concept_id,
    try_to_timestamp(address.address.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(address.address.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT ind.individual_entity_id, explode(address_history) AS address 
  FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all  ind
) AS exploded_addresses
LIMIT 1000;

SELECT * FROM individual_addresses;



-- 5. Create the table of individual_entity_id and associated Addresses
CREATE OR REPLACE TEMP VIEW exp_communication_methods AS
  SELECT ind.individual_entity_id, explode(communication_methods) AS communication_method 
  FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all  ind
;

SELECT * FROM exp_communication_methods;


CREATE OR REPLACE TEMP VIEW individual_communication_methods AS
SELECT 
    individual_entity_id, 
    communication_method.communication_method.rank,
    communication_method.communication_method.is_primary_communication_method,
    communication_method.communication_method.value,
    communication_method.communication_method.value_display,
    communication_method.communication_method.value_raw,
    communication_method.communication_method.type_code,
    try_cast(communication_method.communication_method.type_concept_id AS BIGINT) AS type_concept_id,
    communication_method.communication_method.type_raw_code,
    try_cast(communication_method.communication_method.type_raw_concept_id AS BIGINT) AS type_raw_concept_id,
    communication_method.communication_method.use_code,
    try_cast(communication_method.communication_method.use_concept_id AS BIGINT) AS use_concept_id,
    communication_method.communication_method.use_raw_code,
    try_cast(communication_method.communication_method.use_raw_concept_id AS BIGINT) AS use_raw_concept_id,
    communication_method.communication_method.telephony_country_code,
    try_cast(communication_method.communication_method.telephony_country_concept_id AS BIGINT) AS telephony_country_concept_id,
    communication_method.communication_method.telephony_country_raw_code,
    try_cast(communication_method.communication_method.telephony_country_raw_concept_id AS BIGINT) AS telephony_country_raw_concept_id,
    try_to_timestamp(communication_method.communication_method.period_start, 'dd-MM-yyyy') AS period_start,
    try_to_timestamp(communication_method.communication_method.period_end, 'dd-MM-yyyy') AS period_end
FROM (
  SELECT ind.individual_entity_id, explode(communication_methods) AS communication_method 
  FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all  ind
) AS exploded_communication_methods
;

SELECT * FROM individual_communication_methods;





-- 7. Create the  hub_individual table records
INSERT INTO demo_banking_silver.qdp_individuals_all.hub_individual
(individual_entity_id, tennant_id, national_id, period_start, period_end)
SELECT individual_entity_id, tennant_id, national_id, period_start, period_end
FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all  ind
;

SELECT * FROM demo_banking_silver.qdp_individuals_all.hub_individual;

-- 8. Create the  sat_individual table records
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_individual
    ( 
     individual_id, 
     load_datetime, 
     record_source_id,
     data_source_code,
     data_source_concept_id,
     data_source_raw_code,
     data_source_raw_concept_id,
     birth_date,
     birth_datetime,
     birth_date_numeric,
     estimated_birth_date,
     estimated_birth_date_numeric,
     is_multiple_birth,
     multiple_birth_number,
     birth_place_city,
     birth_place_district,
     birth_place_country,
     birth_place_country_code,
     photo,
     is_deceased,
     deceased_datetime,
     deceased_date_numeric,
     gender_code,
     gender_concept_id,
     gender_raw_code,
     gender_raw_concept_id,
     marital_status_code,
     marital_status_concept_id,
     marital_status_raw_code,
     marital_status_raw_concept_id,
     ethnicity_code,
     ethnicity_concept_id,
     ethnicity_raw_code,
     ethnicity_raw_concept_id,
     religion_code,
     religion_concept_id,
     religion_raw_code,
     religion_raw_concept_id,
     occupation_code,
     occupation_concept_id,
     occupation_raw_code,
     occupation_raw_concept_id,
     nationality_code,
     nationality_concept_id,
     nationality_raw_code,
     nationality_raw_concept_id,
     country_of_residence_code,
     country_of_residence_concept_id,
     country_of_residence_raw_code,
     country_of_residence_raw_concept_id,
     primary_address_id,
     annual_income,
     number_of_dependants,
     graduation_date,
     is_employee,
     is_ex_employee,
     is_graduate,
     is_interpreter_required,
     is_employed,
     is_retired,
     is_student,
     is_individual_merged,
     period_start,
     period_end 
    )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.data_source_code,
  q.data_source_concept_id,
  q.data_source_raw_code,
  q.data_source_raw_concept_id,
  q.birth_date AS birth_date,
  q.birth_datetime AS birth_datetime,
  q.birth_date_numeric AS birth_date_numeric,
  q.estimated_birth_date AS estimated_birth_date,
  q.estimated_birth_date_numeric AS estimated_birth_date_numeric,
  q.is_multiple_birth AS is_multiple_birth,
  q.multiple_birth_number AS multiple_birth_number,
  q.birth_place_city AS birth_place_city,
  q.birth_place_district AS birth_place_district,
  q.birth_place_country AS birth_place_country,
  q.birth_place_country_code AS birth_place_country_code,
  q.photo AS photo,
  q.is_deceased AS is_deceased,
  q.deceased_datetime AS deceased_datetime,
  q.deceased_date_numeric AS deceased_date_numeric,
  q.gender_code AS gender_code,
  q.gender_concept_id AS gender_concept_id,
  q.gender_raw_code AS gender_raw_code,
  q.gender_raw_concept_id AS gender_raw_concept_id,
  q.marital_status_code AS marital_status_code,
  q.marital_status_concept_id AS marital_status_concept_id,
  q.marital_status_raw_code AS marital_status_raw_code,
  q.marital_status_raw_concept_id AS marital_status_raw_concept_id,
  q.ethnicity_code AS ethnicity_code,
  q.ethnicity_concept_id AS ethnicity_concept_id,
  q.ethnicity_raw_code AS ethnicity_raw_code,
  q.ethnicity_raw_concept_id AS ethnicity_raw_concept_id,
  q.religion_code AS religion_code,
  q.religion_concept_id AS religion_concept_id,
  q.religion_raw_code AS religion_raw_code,
  q.religion_raw_concept_id AS religion_raw_concept_id,
  q.occupation_code AS occupation_code,
  q.occupation_concept_id AS occupation_concept_id,
  q.occupation_raw_code AS occupation_raw_code,
  q.occupation_raw_concept_id AS occupation_raw_concept_id,
  q.nationality_code AS nationality_code,
  q.nationality_concept_id AS nationality_concept_id,
  q.nationality_raw_code AS nationality_raw_code,
  q.nationality_raw_concept_id AS nationality_raw_concept_id,
  q.country_of_residence_code AS country_of_residence_code,
  q.country_of_residence_concept_id AS country_of_residence_concept_id,
  q.country_of_residence_raw_code AS country_of_residence_raw_code,
  q.country_of_residence_raw_concept_id AS country_of_residence_raw_concept_id,
  q.primary_address_id AS primary_address_id,
  q.annual_income AS annual_income,
  q.number_of_dependants AS number_of_dependants,
  q.graduation_date AS graduation_date,
  q.is_employee AS is_employee,
  q.is_ex_employee AS is_ex_employee,
  q.is_graduate AS is_graduate,
  q.is_interpreter_required AS is_interpreter_required,
  q.is_employed AS is_employed,
  q.is_retired AS is_retired,
  q.is_student AS is_student,
  q.is_individual_merged AS is_individual_merged,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM demo_banking_silver.qdp_from_quantexa_fincrime_demo.qdp_individuals_all q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_individual;


-- 9. Create the  sat_human_name table records
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_human_name
    ( 
     individual_id, 
     load_datetime, 
     record_source_id,
     rank,
     is_trusted,
     is_preferred,
     use_code,
     use_concept_id,
     use_raw_code,
     use_raw_concept_id,
     initials,
     given1,
     given1_soundex,
     given1_raw,
     given1_raw_soundex,
     given2,
     given2_soundex,
     given2_raw,
     given2_raw_soundex,
     given3,
     given3_soundex,
     given3_raw,
     given3_raw_soundex,
     given4,
     given4_soundex,
     given4_raw,
     given4_raw_soundex,
     given5,
     given5_soundex,
     given5_raw,
     given5_raw_soundex,
     family,
     family_soundex,
     family_raw,
     family_raw_soundex,
     full_given,
     full_given_soundex,
     full_name,
     full_name_soundex,
     full_name_raw,
     full_name_raw_soundex,
     full_name_parsed,
     period_start,
     period_end 
    )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.rank AS rank,
  q.is_trusted AS is_trusted,
  q.is_preferred AS is_preferred,
  q.use_code,
  q.use_concept_id,
  q.use_raw_code,
  q.use_raw_concept_id,
  q.initials,
  q.given1,
  q.given1_soundex,
  q.given1_raw,
  q.given1_raw_soundex,
  q.given2,
  q.given2_soundex,
  q.given2_raw,
  q.given2_raw_soundex,
  q.given3,
  q.given3_soundex,
  q.given3_raw,
  q.given3_raw_soundex,
  q.given4,
  q.given4_soundex,
  q.given4_raw,
  q.given4_raw_soundex,
  q.given5,
  q.given5_soundex,
  q.given5_raw,
  q.given5_raw_soundex,
  q.family,
  q.family_soundex,
  q.family_raw,
  q.family_raw_soundex,
  q.full_given,
  q.full_given_soundex,
  q.full_name,
  q.full_name_soundex,
  q.full_name_raw,
  q.full_name_raw_soundex,
  q.full_name_parsed,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM individual_human_names q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_human_name;


-- 10. Create the  sat_communication_method table records
INSERT INTO demo_banking_silver.qdp_individuals_all.sat_communication_method
    ( 
     individual_id, 
     load_datetime, 
     record_source_id,
     rank,
     is_primary_communication_method,
     value,
     value_display,
     value_raw,
     type_code,
     type_concept_id,
     type_raw_code,
     type_raw_concept_id,
     use_code,
     use_concept_id,
     use_raw_code,
     use_raw_concept_id,
     telephony_country_code,
     telephony_country_concept_id,
     telephony_country_raw_code,
     telephony_country_raw_concept_id,
     period_start,
     period_end
    )
SELECT
  h.individual_id AS individual_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.rank AS rank,
  q.is_primary_communication_method AS is_primary_communication_method,
  q.value AS value,
  q.value_display AS value_display, 
  q.value_raw AS value_raw, 
  q.type_code AS type_code,
  q.type_concept_id AS type_concept_id,
  q.type_raw_code AS type_raw_code,
  q.type_raw_concept_id AS type_raw_concept_id,
  q.use_code AS use_code,
  q.use_concept_id AS use_concept_id,
  q.use_raw_code AS use_raw_code,
  q.use_raw_concept_id AS use_raw_concept_id,
  q.telephony_country_code AS telephony_country_code,
  q.telephony_country_concept_id AS telephony_country_concept_id,
  q.telephony_country_raw_code AS telephony_country_raw_code,
  q.telephony_country_raw_concept_id AS telephony_country_raw_concept_id,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM individual_communication_methods q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id;
  
SELECT * FROM demo_banking_silver.qdp_individuals_all.sat_communication_method;


-- 11. Create the  link_individual_address_history table records    individual_addresses
INSERT INTO demo_banking_silver.qdp_links_all.link_individual_address_history
    ( 
     individual_id, 
     individual_entity_id,
     address_id,
     address_entity_id,
     is_primary_address,
     use_code,
     use_concept_id,
     use_raw_code,
     use_raw_concept_id,
     period_start,
     period_end 
    )
SELECT
  h.individual_id AS individual_id,
  q.individual_entity_id AS individual_entity_id,
  ha.address_id AS address_id,
  q.address_entity_id AS address_entity_id,
  q.is_primary_address AS is_primary_address,
  q.use_code AS use_code,
  q.use_concept_id AS use_concept_id,
  q.use_raw_code AS use_raw_code,
  q.use_raw_concept_id AS use_raw_concept_id,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM individual_addresses q
JOIN demo_banking_silver.qdp_individuals_all.hub_individual h
  ON h.individual_entity_id = q.individual_entity_id
JOIN demo_banking_silver.qdp_locations_all.hub_address ha
  ON ha.address_entity_id = q.address_entity_id;

SELECT * FROM demo_banking_silver.qdp_links_all.link_individual_address_history;



**/



In [0]:

SELECT 
    h.address_id,
    h.address_entity_id,
    s.postal_code,
    s.full_address,
    s.type_code,
    s.data_source_code as how_known_to_network,
    s.house_number,
    s.house_name,
    s.building_number,
    s.building_name,
    s.address_line1,
    s.address_line2,
    s.address_line3,
    s.address_line4,
    s.address_line5,
    s.address_line6,
    s.address_line7,
    s.address_line8,
    s.street,
    s.district,
    s.city,
    s.county,
    s.state,
    s.country,
    s.country_code,
    s.country_concept_id,
    s.country_raw_code,
    s.country_raw_concept_id,
    s.uprn,
    s.what3words,
    s.latitude,
    s.longitude,
    s.directions,
    s.type_concept_id,
    s.type_raw_code,
    s.type_raw_concept_id,
    s.period_start AS sat_address_period_start,
    s.period_end AS sat_address_period_end,
    s.load_datetime,
    s.record_source_id,
    s.data_source_concept_id,
    s.data_source_raw_code,
    s.data_source_raw_concept_id

    FROM demo_banking_silver.qdp_locations_all.hub_address h
    JOIN demo_banking_silver.qdp_locations_all.sat_address s
      ON h.address_id = s.address_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
      WHERE t.tennant_id = 100
    ORDER BY s.postal_code
;

